# 个性化故事生成器：Qwen3-4B QLoRA训练

课程作业快速版。请先在 Colab 菜单中选择 **运行时 → 更改运行时类型 → T4 GPU**，然后按顺序运行全部单元格。训练结果会保存到 Google Drive 的 `storygen_qwen3_lora_20261029` 文件夹。

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '没有检测到GPU，请把Colab运行时改成T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git /content/LLaMA-Factory
%cd /content/LLaMA-Factory
!pip install -q -e .
!pip install -q -U 'bitsandbytes>=0.46.1'

## 上传训练数据
运行下一格后，选择项目目录里的 `storygen_colab_bundle.zip`。

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
!rm -rf /content/storygen_bundle
!unzip -q "$zip_name" -d /content/storygen_bundle
print('已解压:', zip_name)

In [ ]:
from pathlib import Path
import json, shutil
bundle = Path('/content/storygen_bundle')
data_dir = Path('/content/LLaMA-Factory/data/storygen')
data_dir.mkdir(parents=True, exist_ok=True)
for name in ['train.jsonl', 'validation.jsonl', 'test.jsonl', 'stats.json']:
    source = next(bundle.rglob(name))
    shutil.copy2(source, data_dir / name)
dataset_info = next(bundle.rglob('dataset_info.json'))
shutil.copy2(dataset_info, data_dir / 'dataset_info.json')
config = next(bundle.rglob('qwen3_4b_qlora_colab.yaml'))
shutil.copy2(config, '/content/LLaMA-Factory/qwen3_4b_qlora_colab.yaml')
for split in ['train', 'validation', 'test']:
    count = sum(1 for line in open(data_dir / f'{split}.jsonl', encoding='utf-8') if line.strip())
    print(split, count)
print(json.load(open(data_dir / 'stats.json', encoding='utf-8')))

## 开始训练
默认使用1200条训练样本、训练1轮。实测T4训练约28分钟，具体时间取决于Colab分配的运行环境。

In [ ]:
%cd /content/LLaMA-Factory
!llamafactory-cli train qwen3_4b_qlora_colab.yaml

## 测试微调模型
下一格会重新加载4-bit基座模型和Drive中的LoRA权重，并生成一个故事。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
base_model = 'Qwen/Qwen3-4B-Instruct-2507'
adapter_path = '/content/drive/MyDrive/storygen_qwen3_lora_20261029'
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(base_model, quantization_config=quant, device_map='auto')
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()
messages = [
    {'role': 'system', 'content': '你是一名中文短故事作者。自然使用全部关键词。主角的姓名、身份和性格在全文中不得改变；如果主角是人类，不得将其写成机器人。故事应有开端、发展、转折和结局，不要增加复杂亲属关系。'},
    {'role': 'user', 'content': '关键词：雨夜、旧车站、机器人、重逢\n主角：陈安，人类，沉默善良的机械修理师；陈安始终是人类，不是机器人。\n配角：阿七，一台旧型服务机器人。\n目标：陈安希望修复阿七丢失的记忆。\n类型：温暖科幻\n要求：人物身份前后一致，结局回应重逢主题。\n篇幅：300字左右'}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors='pt').to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=500, do_sample=True, temperature=0.65, top_p=0.85, repetition_penalty=1.15)
story = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
print(story)
keywords = ['雨夜', '旧车站', '机器人', '重逢']
missing = [word for word in keywords if word not in story]
print('\n--- 自动检查 ---')
print('关键词覆盖率:', f'{(len(keywords)-len(missing))/len(keywords):.0%}')
print('遗漏关键词:', '无' if not missing else '、'.join(missing))
print('实际字数:', len(story))
print('角色身份冲突:', '是' if ('机器人陈安' in story or '陈安是机器人' in story) else '否')

## 启动课程演示网页
运行下一格会创建一个临时Gradio分享链接。页面关闭或Colab断开后链接会失效。

In [ ]:
!pip install -q 'gradio>=5,<7'
import re, gradio as gr

def generate_personalized_story(keywords, name, identity, personality, goal, genre, tone, length):
    keyword_list = [x.strip() for x in re.split(r'[,，、\s]+', keywords) if x.strip()][:8]
    if not keyword_list:
        return '', '请至少输入一个关键词。'
    prompt = (
        f'关键词：{"、".join(keyword_list)}\n故事类型：{genre}\n情感基调：{tone}\n'
        f'主角：{name}\n身份：{identity}\n性格：{personality}\n目标：{goal}\n'
        f'篇幅：约{int(length)}字\n要求：全部关键词自然出现；主角身份前后一致；结局回应主角目标。'
    )
    chat = [
        {'role': 'system', 'content': '你是一名中文短故事作者。严格遵守人物设定；如果主角是人类，不得写成机器人。写出开端、发展、转折和结局，不要增加复杂亲属关系。'},
        {'role': 'user', 'content': prompt},
    ]
    rendered = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    batch = tokenizer([rendered], return_tensors='pt').to(model.device)
    with torch.no_grad():
        generated = model.generate(**batch, max_new_tokens=min(max(int(length)*2, 300), 900), do_sample=True, temperature=0.65, top_p=0.85, repetition_penalty=1.15)
    result = tokenizer.decode(generated[0][batch.input_ids.shape[1]:], skip_special_tokens=True).strip()
    missing = [word for word in keyword_list if word not in result]
    coverage = (len(keyword_list)-len(missing))/len(keyword_list)
    conflict = f'机器人{name}' in result or f'{name}是机器人' in result
    report = f'关键词覆盖率：{coverage:.0%}\n实际字数：{len(result)}\n遗漏关键词：{"无" if not missing else "、".join(missing)}\n角色身份冲突：{"是" if conflict else "否"}'
    return result, report

with gr.Blocks(title='个性化故事生成器') as story_demo:
    gr.Markdown('# 个性化故事生成器\n输入关键词和人物设定，使用微调后的Qwen3-4B生成中文短故事。')
    with gr.Row():
        with gr.Column():
            ui_keywords = gr.Textbox(label='关键词', value='雨夜，旧车站，机器人，重逢')
            ui_name = gr.Textbox(label='角色姓名', value='陈安')
            ui_identity = gr.Textbox(label='角色身份', value='人类机械修理师')
            ui_personality = gr.Textbox(label='角色性格', value='沉默、善良、谨慎')
            ui_goal = gr.Textbox(label='角色目标', value='修复旧机器人阿七丢失的记忆')
            ui_genre = gr.Dropdown(['童话','科幻','校园','悬疑','都市','冒险'], value='科幻', label='故事类型')
            ui_tone = gr.Dropdown(['温暖','紧张','幽默','悲伤','治愈'], value='温暖', label='情感基调')
            ui_length = gr.Slider(200, 600, value=300, step=50, label='目标字数')
            ui_button = gr.Button('生成故事', variant='primary')
        with gr.Column():
            ui_story = gr.Textbox(label='生成结果', lines=18)
            ui_report = gr.Textbox(label='自动质检', lines=5)
    ui_button.click(generate_personalized_story, [ui_keywords, ui_name, ui_identity, ui_personality, ui_goal, ui_genre, ui_tone, ui_length], [ui_story, ui_report])

story_demo.launch(share=True, debug=False)